# Turn a finished run into a results folder

Point `RUNS` at the `runs/<tag>/` folder that came back from the cluster and run
the notebook. It writes

    results/<dataset>_<date>/
        figures/  pooled_r2_by_featureset, lolo_by_held_out_<group>, kraken_space
        tables/   summary, metrics_by_split, predictions, lolo_report
        report.json

and shows each figure inline on the way past.

**Everything about how these figures look lives in one file: `gpc/figures.py`.**
Titles, axis labels, colours, sizes, label positions, axis limits. Change it
there and re-run this notebook; nothing else draws anything that gets saved.

No PyTorch needed -- this is review only.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()          # gp_collab_hub/
sys.path.insert(0, str(ROOT))

from gpc import figures
from gpc.config import bundle_group, bundle_name, read_json
from gpc.report import dataset_ligands, kraken_population, write_report
from gpc.results import load_results, lolo_report, prepare

pd.set_option("display.width", 200, "display.max_columns", 60)

# ---- Editable ------------------------------------------------------------
DATASET = "perera"                       # datasets/<DATASET>/inputs
RUNS    = ROOT / "runs" / "perera_20260923"   # what came back from the cluster
DATE    = None                           # None -> today, else "YYYYMMDD"
NAME    = None                           # None -> the bundle's display_name
METRIC  = "r2"                           # r2 | rmse | mae | kendall_tau
FORMATS = ("png",)                       # add "pdf" or "svg" for publication
REFRESH = False                          # True after rerunning or adding tasks
# --------------------------------------------------------------------------

BUNDLE = ROOT / "datasets" / DATASET / "inputs"
prepared = read_json(BUNDLE / "config.json")
GROUP = bundle_group(prepared)
DISPLAY_NAME = NAME or bundle_name(prepared)
print(f"{DISPLAY_NAME}: {prepared['data']['expected_rows']} rows, "
      f"{prepared['data'].get('expected_groups')} {GROUP}s")
print(f"runs   {RUNS}")

## 1. Load the run

`prepare` builds the collected review tables the first time and reuses them
afterwards; pass `REFRESH=True` above once you have added or rerun tasks.
`complete` is False when any task in the registry has no output -- read the
figures below with that in mind if it is.

In [ ]:
EXPORT = prepare(RUNS, bundle=BUNDLE, refresh=REFRESH)
tables, collection = load_results(EXPORT)

print("complete:", collection["complete"],
      f"({collection['completed_tasks']}/{collection['expected_tasks']} tasks)")

pooled = tables["summary"][tables["summary"].aggregation == "pooled_predictions"]
display(pooled.pivot(index="model", columns="method", values=["r2", "rmse"]).round(3))

## 2. The figures, one at a time

Drawn here so you can look before anything is written. These are the exact
functions `write_report` calls, so what you see is what lands in the folder.

In [ ]:
plt.close("all")
fig_pooled = figures.pooled_by_featureset(tables["summary"], DISPLAY_NAME, METRIC)
display(fig_pooled)
plt.close(fig_pooled)

In [ ]:
plt.close("all")
fig_lolo = figures.lolo_by_group(tables["metrics_by_split"], DISPLAY_NAME, METRIC, GROUP)
display(fig_lolo)
plt.close(fig_lolo)

### Where this dataset's ligands sit in the Kraken space

The grey cloud is the whole 1,223-ligand Kraken reference, read from
`reference/kraken/` so it is identical in every dataset's report. Left panel is
PC1 vs PC2 of the frozen reference PCA; right panel is the two buried-volume
descriptors that `selected_2` is built from.

Point labels are placed automatically and then pushed apart until their drawn
boxes stop overlapping. To pin one by hand, add it to
`STYLE["kraken"]["offsets"]` in `gpc/figures.py`, in points:
`{"XPhos": (18, -14)}`.

In [ ]:
plt.close("all")
population = kraken_population(BUNDLE)
used = dataset_ligands(BUNDLE, population)
display(used[["name", "kraken_id", "PC1", "PC2", "vbur_pct_boltz", "vbur_pct_min"]].round(2))

fig_space = figures.kraken_space(population, used, DISPLAY_NAME)
display(fig_space)
plt.close(fig_space)

## 3. LOLO three ways

`per_ligand` scores each held-out group on its own rows, `mean_across_ligands`
averages those, and `pooled_over_folds` concatenates every fold and scores once.

Pooled and averaged are **not** interchangeable. A per-group R^2 is measured
against that group's own, smaller variance, so the averaged figure is
systematically harsher than the pooled one. The bar chart above is the per-group
view; the pooled figures are the ones in the feature-section chart.

In [ ]:
report = lolo_report(tables["predictions"], tables["metrics_by_split"])
display(report[["model", "view", "scope", "n_test", "r2", "rmse", "mae",
                "kendall_tau"]].round(4))

## 4. Write the folder

One call, and it redraws everything from scratch -- so this is also the cell to
re-run after editing `gpc/figures.py`.

In [ ]:
destination = write_report(RUNS, BUNDLE, date=DATE, dataset_name=NAME,
                           metric=METRIC, formats=FORMATS, refresh=False)

display(pd.DataFrame([{"file": str(p.relative_to(destination)),
                       "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(destination.rglob("*")) if p.is_file()]))

## 5. Optional: feature importance (ARD)

Needs a run fitted with `ard: true`. `relevance` is 1/lengthscale -- an RBF
varies fastest along its shortest lengthscales, so a large relevance means the
fit leans on that column, and a feature pushed out to a huge lengthscale has
effectively been switched off.

The ranking holds **within one section**. Numeric descriptors are standardized
per fold while one-hots stay raw 0/1, and two sections' kernels share no scale,
so compare the ordering and not the magnitudes.

This is exploratory and is not part of the saved report; `compare_datasets.ipynb`
has the across-datasets version.

In [ ]:
from gpc.results import plot_feature_importance, rank_features

IMPORTANCE_MODEL = "selected_5"    # group_ohe, selected_2, selected_5, pc_top, pc_scores
GROUP_ONLY       = True            # True -> ligand descriptors only
TOP_N            = 20

plt.close("all")
try:
    display(rank_features(RUNS, BUNDLE, IMPORTANCE_MODEL, method="lolo",
                          ligand_only=GROUP_ONLY, top=TOP_N).round(4))
    fig_importance = plot_feature_importance(RUNS, BUNDLE, IMPORTANCE_MODEL,
                                             method="lolo", ligand_only=GROUP_ONLY,
                                             top=TOP_N)
    display(fig_importance)
    plt.close(fig_importance)
except ValueError as exc:
    # An isotropic run has one lengthscale for the whole block, so there is
    # nothing per-feature inside it. Say so rather than invent a number.
    print(f"{IMPORTANCE_MODEL}: {exc}")